<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 15 - Procesamiento de Datos: MapReduce y Apache Spark

**Módulo:** Big Data Aplicado (BDA - 5075)

**Contenido:** 1.2 Procesamiento de datos — Paradigmas (MapReduce, Spark, Flink), herramientas, pipelines y PySpark en práctica

---

## Objetivos
- Comprender los paradigmas de procesamiento de datos (batch vs streaming)
- Diferenciar MapReduce, Spark y Flink: cuándo usar cada uno
- Conocer los formatos de archivo para Big Data (Parquet, Avro, ORC)
- Implementar pipelines de procesamiento con PySpark

## Instalación en Google Colab

In [ ]:
# Instalar PySpark (ejecutar solo en Google Colab)
!pip install pyspark -q

## Paradigmas de Procesamiento de Datos

### Evolución histórica

```
2004: MapReduce (Google)
  └─ Procesamiento distribuido batch, escribe a disco entre fases

2006: Hadoop
  └─ Implementación open source de MapReduce (HDFS + YARN)

2014: Apache Spark
  └─ Procesamiento en memoria, 10-100x más rápido que MapReduce

2016: Apache Flink
  └─ Streaming nativo de primera clase, procesamiento unificado

2020+: Procesamiento unificado batch + streaming
  └─ Delta Lake, Apache Iceberg, Databricks
```

### MapReduce: el origen

**Fases:** `Map → Shuffle → Reduce`

```
Datos → [Map₁, Map₂, Map₃] → Shuffle → [Reduce₁, Reduce₂] → Resultado
```

**Limitaciones:** escribe a disco entre fases (lento), difícil para algoritmos iterativos (ML).

### Comparativa de herramientas

| Herramienta | Tipo | Latencia | Throughput | Curva aprendizaje |
|-------------|------|----------|------------|-------------------|
| **Hadoop MR** | Batch | Alta | Alto | Media |
| **Spark** | Batch/µBatch | Media | Muy alto | Media |
| **Flink** | Streaming nativo | Muy baja | Alto | Alta |
| **Kafka Streams** | Streaming | Baja | Alto | Media |

### Formatos de archivo para Big Data

| Formato | Tipo | Compresión | Schema | Uso ideal |
|---------|------|------------|--------|-----------|
| CSV | Row | Baja | No | Intercambio de datos |
| JSON | Row | Baja | Flexible | APIs REST |
| **Parquet** | Column | Alta | Sí | Analytics (recomendado) |
| Avro | Row | Media | Sí | Streaming (Kafka) |
| ORC | Column | Alta | Sí | Hive / Hadoop |

> **¿Por qué Parquet?** Lee solo las columnas necesarias, compresión eficiente y schema embebido. Es el estándar de facto en el ecosistema Big Data.

---

## Tutorial PySpark: Apache Spark en práctica

## 1. Tutorial completo de PySpark

In [ ]:
# ========================================
# IMPORTACIONES Y CONFIGURACIÓN
# ========================================
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, count, min, max
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
import pandas as pd

# Crear SparkSession
spark = SparkSession.builder \
    .appName("TutorialPySpark") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

In [ ]:
# ========================================
# CREAR DATAFRAMES
# ========================================
print("=== CREAR DATAFRAMES ===")

# Desde lista de diccionarios
df = spark.createDataFrame([
    {"id": 1, "valor": 100, "categoria": "A"},
    {"id": 2, "valor": 200, "categoria": "B"},
    {"id": 3, "valor": 150, "categoria": "A"},
    {"id": 4, "valor": 300, "categoria": "B"},
    {"id": 5, "valor": 250, "categoria": "A"}
])

print("DataFrame creado:")
df.show()

In [ ]:
# ========================================
# OPERACIONES BÁSICAS
# ========================================
print("=== OPERACIONES BÁSICAS ===")

# Ver esquema
print("Esquema:")
df.printSchema()

# Mostrar datos
print(f"\nTotal filas: {df.count()}")

# Seleccionar columnas
print("\nSeleccionar id y valor:")
df.select("id", "valor").show()

# Seleccionar con transformación
print("Valor duplicado:")
df.select(col("id"), (col("valor") * 2).alias("valor_doble")).show()

# Filtrar
print("Filtrar valor > 150:")
df.filter(col("valor") > 150).show()

# Ordenar
print("Ordenar por valor descendente:")
df.orderBy(col("valor").desc()).show()

In [ ]:
# ========================================
# AGREGACIONES
# ========================================
print("=== AGREGACIONES ===")

# Agregación simple
print("Conteo por categoría:")
df.groupBy("categoria").count().show()

# Múltiples agregaciones
print("\nMúltiples agregaciones por categoría:")
df.groupBy("categoria").agg(
    sum("valor").alias("total"),
    avg("valor").alias("promedio"),
    count("*").alias("registros"),
    min("valor").alias("minimo"),
    max("valor").alias("maximo")
).show()

In [ ]:
# ========================================
# FUNCIONES DE VENTANA
# ========================================
print("=== FUNCIONES DE VENTANA ===")

print("Ranking por valor dentro de cada categoría:")
window = Window.partitionBy("categoria").orderBy(col("valor").desc())
df.withColumn("ranking", row_number().over(window)).show()

In [ ]:
# ========================================
# SPARK SQL
# ========================================
print("=== SPARK SQL ===")

# Registrar DataFrame como tabla temporal
df.createOrReplaceTempView("datos")

# Ejecutar SQL
print("Consulta SQL - Total por categoría:")
resultado = spark.sql("""
    SELECT 
        categoria, 
        SUM(valor) as total,
        AVG(valor) as promedio
    FROM datos
    GROUP BY categoria
    ORDER BY total DESC
""")
resultado.show()

print("\n✅ Tutorial PySpark completado!")

---

## Recursos adicionales

- [PySpark Documentation](https://spark.apache.org/docs/latest/api/python/)
- [Spark SQL Guide](https://spark.apache.org/docs/latest/sql-programming-guide.html)